In [16]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib

df=pd.read_csv('../data/breast-cancer.csv')
display(df.head())

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [17]:
from multiprocessing.reduction import duplicate


print(df.info())
print(df.describe())
print(df.isnull().sum())
duplicate=df.duplicated().sum()
df.drop_duplicates(inplace=True)


X=df.drop(['diagnosis'], axis=1)
y=df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/scaler.pkl')

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", np.sum(pca.explained_variance_ratio_))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [18]:
joblib.dump(pca, '../models/pca.pkl')
print("PCA model saved to ../models/pca.pkl")

PCA model saved to ../models/pca.pkl


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
accuracy_score, classification_report, confusion_matrix,
f1_score, precision_score, recall_score
)
from sklearn.model_selection import cross_val_score,GridSearchCV


model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_pca, y_train)
predictions = model.predict(X_test_pca)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Classification Report:\n", classification_report(y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))
print("F1 Score:", f1_score(y_test, predictions, pos_label='M'))
print("Precision Score:", precision_score(y_test, predictions, pos_label='M'))
print("Recall Score:", recall_score(y_test, predictions, pos_label='M'))

cross_val_scores = cross_val_score(model, X_train_pca, y_train, cv=5)
print("Cross-validation scores:", cross_val_scores)

param_grid = {'C': [0.1, 1, 10],"solver":['liblinear', 'lbfgs']}
grid_search = GridSearchCV(model, param_grid=param_grid, cv=5)
grid_search.fit(X_train_pca, y_train)
print("Best parameters from grid search:", grid_search.best_params_)
print("Best cross-validation score from grid search:", grid_search.best_score_)

print("accuracy after parameter tuning:", grid_search.score(X_test_pca, y_test))

Accuracy: 0.9912280701754386
Classification Report:
               precision    recall  f1-score   support

           B       0.99      1.00      0.99        71
           M       1.00      0.98      0.99        43

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114

Confusion Matrix:
 [[71  0]
 [ 1 42]]
F1 Score: 0.9882352941176471
Precision Score: 1.0
Recall Score: 0.9767441860465116
Cross-validation scores: [0.92307692 0.96703297 0.95604396 0.94505495 0.93406593]
Best parameters from grid search: {'C': 1, 'solver': 'liblinear'}
Best cross-validation score from grid search: 0.9472527472527472
accuracy after parameter tuning: 0.9912280701754386


In [20]:
best_model = grid_search.best_estimator_

patient=np.array([[17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
 1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
 25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189,0.3403]])
patient_scaled = scaler.transform(patient)
load_pca = joblib.load('../models/pca.pkl')
patient_pca = load_pca.transform(patient_scaled)
prediction = best_model.predict(patient_pca)
probability = best_model.predict_proba(patient_pca)
print("Probability of being malignant:", probability[0][1])
print("\nPrediction")

if prediction[0] == "M":

    print("Malignant Tumor")

else:

    print("Benign Tumor")

print("\nPrediction Probability")

print(probability)



Probability of being malignant: 0.0

Prediction
Benign Tumor

Prediction Probability
[[1. 0.]]


c:\Users\Shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [21]:
joblib.dump(pca, '../models/pca.pkl')
print("PCA model saved to ../models/pca.pkl")

PCA model saved to ../models/pca.pkl
